# vessel-watch: detector and two-stage pipeline

Launcher only; all logic lives in the `vesselwatch` package. Three parts:

1. Train a YOLO11-OBB detector on the dataset's oriented boxes and score it once on the held-out test split.
2. Run the two-stage pipeline (detect, cut out each vessel along its box, classify with the ResNet101 from the crop study) at label levels 1, 2 and 3, scoring each stage against the ground truth.
3. Run two comparisons at level 2: the first prototype's setup (stock detector, upright crops, the scene-trained ResNet101), and the stock detector with the new classifier.

Needs the crop study's runs on Drive (step 4a). Runtime > Change runtime type > GPU before running. If the session drops, run all cells again: detector training continues from its last epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

MODELS_DIR = '/content/drive/MyDrive/5.Projects/models'  # weights of the original study
DATA_DIR_DRIVE = '/content/drive/MyDrive/5.Projects/Data/ShipRSImageNet_V1'
DATA_DIR_LOCAL = '/content/data/ShipRSImageNet_V1'
RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'      # results are written straight to Drive
CACHE_DIR = '/content/cache'                             # exported detector dataset, rebuilt each session

DETECTOR_CONFIG = 'configs/detect/yolo11s_obb.yaml'
DETECTOR = f'{RUNS_DIR}/yolo11s_obb/weights/best.pt'

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib, pyyaml and opencv
%pip install -q --no-deps -e /content/vessel-watch
%pip install -q ultralytics
%cd /content/vessel-watch
!git log --oneline -1

## Data

Reading images from Drive during training is slow, so they are copied to the Colab disk once per session.

In [ ]:
import shutil

if os.path.exists(DATA_DIR_LOCAL):
    print('Local copy already exists')
else:
    os.makedirs(os.path.dirname(DATA_DIR_LOCAL), exist_ok=True)
    shutil.copytree(DATA_DIR_DRIVE, DATA_DIR_LOCAL)
    print('Copied dataset to', DATA_DIR_LOCAL)

os.environ['VESSELWATCH_DATA'] = DATA_DIR_LOCAL
os.environ['VESSELWATCH_RUNS'] = RUNS_DIR
os.environ['VESSELWATCH_CACHE'] = CACHE_DIR

## 1. Train the detector

Writes to `RUNS_DIR/yolo11s_obb/`. The last line gives precision, recall and mAP on the test split.

In [ ]:
!python -m vesselwatch.detect.yolo --config $DETECTOR_CONFIG

## 2. Two-stage pipeline at each label level

For every vessel in the test split: was it found (a detection overlapping its box with IoU of at least 0.5), and was the found vessel given the right class? "Classifier on true boxes" is the crop study's test accuracy, the ceiling the pipeline could reach with a perfect detector.

In [ ]:
for level in (1, 2, 3):
    !python -m vesselwatch.detect.pipeline --name two_stage_level{level} --detector "$DETECTOR" --classifier-config configs/crops/level{level}_resnet101.yaml --classifier-weights "$RUNS_DIR/crops_level{level}_resnet101/best.pt"
    print()

## 3. Comparisons at level 2

First the setup of the original prototype: a stock YOLO11n-OBB that never saw this dataset (only its `ship` class is kept), upright crops stretched to a square, and the ResNet101 that was trained on whole scenes. Then the same stock detector with the new classifier, to show how much each stage contributes.

In [ ]:
!python -m vesselwatch.detect.pipeline --name prototype_level2 --detector yolo11n-obb.pt --keep-class ship --conf 0.5 --crop upright --margin 0 --preprocess squash --classifier-config configs/legacy/level2_resnet101.yaml --classifier-weights "$MODELS_DIR/best_model_ResNet_Level2.pth"
print()
!python -m vesselwatch.detect.pipeline --name stock_detector_level2 --detector yolo11n-obb.pt --keep-class ship --classifier-config configs/crops/level2_resnet101.yaml --classifier-weights "$RUNS_DIR/crops_level2_resnet101/best.pt"